# 01 - Data, Baseline and First Explainability Maps

This notebook establishes the experimental substrate for the explainability study. It consolidates data loading, baseline availability and first attribution analysis into a single workflow.

Technical scope:

1. **Dataset interface validation**  
   Verify that AwA2 images, labels, class names and splits are loaded consistently through the manifest-based DataLoader.
2. **Baseline classifier availability**  
   Load or train the ResNet50 classifier used as the explained model in all later experiments.
3. **Initial attribution extraction**  
   Inspect Grad-CAM and Integrated Gradients on selected correct and incorrect predictions.
4. **Gradient-level diagnostics**  
   Optionally compute tensor statistics for logits, probabilities, gradients, activations and normalized saliency maps.

The notebook is not a generic training notebook. Its purpose is to prepare and inspect the model that will later be stress-tested for explanation faithfulness.


In [ ]:
from pathlib import Path
from IPython.display import Image, display, HTML
import csv
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists() and (candidate / "scripts").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the project root containing src/ and scripts/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MANIFEST = PROJECT_ROOT / "data" / "AWA2_subset_background20" / "awa2_manifest_subset.csv"
METADATA_ROOT = PROJECT_ROOT / "data" / "AWA2"
CHECKPOINT = PROJECT_ROOT / "outputs" / "checkpoints" / "best_resnet50_awa2.pt"

print("project_root:", PROJECT_ROOT)
print("manifest:", MANIFEST, "exists=", MANIFEST.exists())
print("checkpoint:", CHECKPOINT, "exists=", CHECKPOINT.exists())

# Set these flags to True only when you intentionally want to recompute outputs.
RUN_TRAINING = False
RUN_XAI = CHECKPOINT.exists()
RUN_GRADIENT_DIAGNOSTICS = False


## 1. Dataset and Manifest

A **manifest** is the CSV file that tells the project where every image is and which label it belongs to. Instead of scanning directories every time, the DataLoader reads this table.

Expected manifest columns:

- `filepath`: image location
- `label`: integer class id
- `class_name`: human-readable class name
- `split`: `train`, `val` or `test`

The DataLoader applies the standard ResNet/ImageNet preprocessing pipeline:

```text
Resize / crop -> convert to tensor -> normalize with ImageNet mean and std
```

This matters because ResNet50 was originally trained on ImageNet. If normalization is wrong, predictions and gradients become hard to interpret.


In [ ]:
from src.data import build_dataloaders, infer_class_map_path, denormalize_batch

CLASS_MAP = infer_class_map_path(MANIFEST)
loaders = build_dataloaders(
    manifest_path=MANIFEST,
    class_map_path=CLASS_MAP,
    batch_size=8,
    num_workers=0,
    pin_memory=False,
)

for split, loader in loaders.items():
    dataset = loader.dataset
    print(f"{split}: {len(dataset)} images | visible_classes={len(dataset.visible_classes)} | mapped_classes={len(dataset.classes)}")

images, labels, class_names, paths = next(iter(loaders['train']))
print('batch tensor shape:', images.shape)
print('label tensor shape:', labels.shape)
print('first class names:', list(class_names[:5]))
print('first image path:', paths[0])


### How to read this output

A correct batch should have shape similar to:

```text
[batch_size, 3, 224, 224]
```

That means each image has 3 RGB channels and has been transformed to the `224 x 224` input expected by ResNet50. The number of mapped classes should match the subset currently used by the project.


## 2. Baseline Model

The baseline classifier is **ResNet50**, loaded from `torchvision` and adapted to the AwA2 classes by replacing the final linear layer.

Conceptually:

```text
image -> ResNet convolutional backbone -> feature vector -> linear classifier -> animal class
```

The checkpoint stored in `outputs/checkpoints/best_resnet50_awa2.pt` is used by all later phases. XAI maps, stress tests, TCAV and Concept Bottleneck comparisons only make sense if this baseline exists.


In [ ]:
if RUN_TRAINING:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'training' / 'train_baseline.py'),
        '--manifest', str(MANIFEST),
        '--checkpoint-output', str(CHECKPOINT),
        '--epochs', '5',
        '--batch-size', '32',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('training skipped')
    print('checkpoint exists:', CHECKPOINT.exists())


## 3. Training History

This cell reads the saved training history. In the current run, validation accuracy peaks early and then slightly decreases while training accuracy keeps increasing. That pattern suggests mild overfitting pressure, which is useful to mention in the final report.


In [ ]:
HISTORY = PROJECT_ROOT / 'outputs' / 'reports' / 'training_history.csv'
if HISTORY.exists():
    with HISTORY.open('r', newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        print(row)
else:
    print('missing:', HISTORY)


### Interpretation checkpoint

Use the best validation epoch as the most credible baseline. If the model overfits, saliency maps can become even more dataset-specific, which strengthens the motivation for stress tests.


## Explainability Framing

A saliency map is a scalar field over the input image. It is usually visualized as a heatmap, where high values mark pixels or regions that a method considers important for a target class score.

For an image tensor `x` and target score `S_c(x)`, attribution asks how much each part of `x` contributes to the score assigned to class `c`. This project keeps two local attribution methods:

- **Grad-CAM**: localizes class evidence in the last convolutional representation. It is spatially coarse because ResNet50 `layer4` has a small feature grid.
- **Integrated Gradients**: attributes the score difference between a blurred baseline and the real image by integrating gradients along the interpolation path.

These maps measure model sensitivity and localization hypotheses. They are not proof that the classifier reasoned semantically, which is why later notebooks apply background perturbations and concept-level diagnostics.


## 4. Local Attribution Comparison

This section generates a direct visual comparison between the two maintained local attribution methods.

- **Grad-CAM** computes gradients of the target score with respect to late feature maps, averages those gradients spatially to obtain channel weights, and combines the weighted feature maps into a heatmap.
- **Integrated Gradients** constructs a path from a blurred version of the same image to the original input. Gradients are accumulated along this path, so the resulting map depends explicitly on the chosen baseline.

Read the output row by row. If a prediction is correct but the heatmap concentrates on background texture, the model may be exploiting context rather than animal morphology. If a prediction is wrong, the heatmap helps inspect which visual evidence supported the incorrect class.


In [ ]:
XAI_FIGURE = PROJECT_ROOT / 'outputs' / 'figures' / 'xai_method_comparison_notebook.png'

if RUN_XAI:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'experiments' / 'run_xai.py'),
        '--manifest', str(MANIFEST),
        '--checkpoint', str(CHECKPOINT),
        '--output', str(XAI_FIGURE),
        '--max-images', '4',
        '--ig-steps', '16',
        '--ig-internal-batch-size', '4',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('XAI generation skipped; set RUN_XAI=True to recompute the local attribution comparison.')

print(XAI_FIGURE)
if XAI_FIGURE.exists():
    display(Image(filename=str(XAI_FIGURE)))


In [ ]:
print(XAI_FIGURE)
if XAI_FIGURE.exists():
    display(Image(filename=str(XAI_FIGURE)))
else:
    print('missing; run the previous cell to generate the comparison figure')


## Optional Attribution Tensor Diagnostics

This diagnostic cell recomputes explanations on a small batch and prints tensor statistics. It is disabled by default because Integrated Gradients requires several forward/backward evaluations.

Relevant checks:

- logits and probabilities must be finite and class-specific;
- raw input gradients can contain positive and negative values before reduction;
- normalized saliency maps must lie in `[0, 1]`;
- Grad-CAM activations should have spatial shape close to `7 x 7` for ResNet50 `layer4`;
- Integrated Gradients attributions should be inspected before reducing RGB channels to a single heatmap.


In [ ]:
if RUN_GRADIENT_DIAGNOSTICS:
    import torch
    from src.model import build_resnet50_classifier, get_device
    from src.xai import (
        gradcam_saliency,
        input_gradient_saliency,
        integrated_gradients_maps,
        log_tensor_stats,
    )

    device = get_device('auto')
    num_classes = len(loaders['train'].dataset.classes)
    model = build_resnet50_classifier(num_classes=num_classes, pretrained=False)
    checkpoint = torch.load(CHECKPOINT, map_location=device)
    model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
    model.to(device).eval()

    batch = next(iter(loaders['test']))
    images = batch[0][:2].to(device)
    labels = torch.as_tensor(batch[1][:2], dtype=torch.long, device=device)
    names = list(batch[2][:2])

    with torch.no_grad():
        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        confidences, predictions = probabilities.max(dim=1)

    print('selected examples')
    for i, name in enumerate(names):
        print(f'[{i}] true={name} target={int(labels[i])} pred={int(predictions[i])} confidence={float(confidences[i]):.4f}')

    log_tensor_stats('diagnostics.images.normalized', images)
    log_tensor_stats('diagnostics.logits', logits)
    log_tensor_stats('diagnostics.probabilities', probabilities)

    input_maps = input_gradient_saliency(model, images, labels)
    target_layer = model.layer4[-1]
    gradcam_maps = gradcam_saliency(model, images, labels, target_layer)
    ig_maps, ig_attributions, ig_baseline = integrated_gradients_maps(
        model, images, labels, steps=8, internal_batch_size=2
    )

    log_tensor_stats('diagnostics.input_gradient_maps_debug', input_maps)
    log_tensor_stats('diagnostics.gradcam_maps', gradcam_maps)
    log_tensor_stats('diagnostics.ig_maps', ig_maps)
    log_tensor_stats('diagnostics.ig_attributions', ig_attributions)
    log_tensor_stats('diagnostics.ig_blurred_baseline', ig_baseline)
else:
    print('Gradient diagnostics skipped; set RUN_GRADIENT_DIAGNOSTICS=True to inspect tensors.')
